# 01 · Professor: rotular as biografias no Colab

Corre com **Runtime → Change runtime type → T4 GPU**.
Produz `labels.jsonl` (o mesmo formato do `label.py` com Bedrock) e envia-o para o S3.
Se o Colab desligar, volta a correr as células 1, 2 e 4: o rótulo retoma onde parou.

## 1. Preparar o ambiente

In [ ]:
# Instalar dependências (awscli: o Colab não traz a CLI da AWS)
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets boto3 pydantic awscli

import os
from google.colab import drive, userdata

# Secrets do Colab (ícone da chave, à esquerda): AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, DATA_BUCKET
for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    os.environ[k] = userdata.get(k)
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

drive.mount("/content/drive")  # tudo o que demora vai para o Drive: sobrevive a desconexões
DRIVE = "/content/drive/MyDrive/wiki-bio-distill"
os.makedirs(DRIVE, exist_ok=True)

# Repositório: público, ou privado com um secret GITHUB_TOKEN
try:
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/pinthoz/wiki-bio-distill.git"
except Exception:
    url = "https://github.com/pinthoz/wiki-bio-distill.git"
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
%cd /content/repo
!git pull -q
!mkdir -p data preds
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Buscar os textos e ligar o ficheiro de saída ao Drive

In [ ]:
!aws s3 cp s3://$DATA_BUCKET/v1/raw.jsonl data/
!wc -l data/raw.jsonl

# labels.jsonl vive no Drive; data/labels.jsonl é só um atalho para lá
!touch {DRIVE}/labels.jsonl
!ln -sf {DRIVE}/labels.jsonl data/labels.jsonl

## 3. Ensaio com 50 exemplos

Vê a taxa de erros, a velocidade e alguns rótulos antes de lançar tudo.

In [ ]:
!python label_local.py --model Qwen/Qwen2.5-7B-Instruct --limit 50

In [ ]:
import json
rows = {r["id"]: r for r in map(json.loads, open("data/raw.jsonl"))}
labels = [json.loads(l) for l in open("data/labels.jsonl")]
print(f"{len(labels)} rotulados, {sum('error' in l for l in labels)} com erro\n")
for l in labels[:5]:
    print(rows[l["id"]]["text"][:200], "…")
    print(json.dumps(l.get("label") or l, ensure_ascii=False, indent=2), "\n")

## 4. Rotular o resto

Para 3000 biografias, conta com uma a duas horas. O script mostra o tempo que falta.

In [ ]:
!python label_local.py --model Qwen/Qwen2.5-7B-Instruct

## 5. Enviar para o S3

In [ ]:
!grep -c '"error"' data/labels.jsonl || true
!aws s3 cp data/labels.jsonl s3://$DATA_BUCKET/v1/labels.jsonl

Depois disto: no teu portátil, `aws s3 cp s3://$BUCKET/v1/labels.jsonl data/`, `python review.py` e `python split.py` (passo 4 da ficha), e envia `train`, `val` e `test` para o S3.